# Join two-page source scans and retain unused images

This notebook converts the complete 1940–2007 full-page image set into an annotation-ready set. It uses the cleaned face-entry CSV to identify one- and two-page source scans, and the companion dropped-entry CSV to identify scans that span more than two pages.

Single-page images remain byte-for-byte unchanged. Each available two-page scan is rendered as one horizontal double-page JPEG. The two original JPEGs are copied to an unused-image folder before being removed from the kept folder; images belonging to scans of three or more pages are moved to that same unused folder. Finally, the kept folder is renamed to `full_pages_1940_2007_joined`.

The transformation requires the original folder and refuses to run if the final joined folder already exists. If JPEG encoding is interrupted before the transfer step, it can resume from the valid joined JPEGs already written beside the source images.

In [ ]:
# Move deliberately excluded images out of the joined-image set.
from pathlib import Path
import re
import shutil

import pandas as pd

MANUALLY_EXCLUDED_FILENAMES = [
    "2006-0610-0118.jpg",
    "2006-0610-0119.jpg",
]

joined_dir = Path("../../data/images/full_pages_1940_2007_joined")
manually_excluded_dir = joined_dir / "manually_excluded_images"
assert joined_dir.exists(), joined_dir
manually_excluded_dir.mkdir(exist_ok=True)

moved_filenames = []
missing_image_filenames = []
for filename in MANUALLY_EXCLUDED_FILENAMES:
    destination = manually_excluded_dir / filename
    source_paths = [
        path
        for path in joined_dir.rglob(filename)
        if path.is_file() and path.parent != manually_excluded_dir
    ]

    if destination.exists():
        assert not source_paths, f"Duplicate sources for already excluded image: {filename}"
        continue

    if not source_paths:
        missing_image_filenames.append(filename)
        continue

    assert len(source_paths) == 1, f"Expected one source image for {filename}, found: {source_paths}"
    shutil.move(source_paths[0], destination)
    assert destination.exists()
    moved_filenames.append(filename)


# Remove face entries whose encoded source-page list references a manually excluded image.
cleaned_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_cleaned.csv")
manual_exclusion_audit_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_cleaned-manually-excluded-face-entries.csv")
filename_pattern = r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)_"

faces = pd.read_csv(cleaned_csv, dtype={"Filename": "string"})
original_columns = list(faces.columns)
filename_parts = faces["Filename"].str.extract(filename_pattern)
assert filename_parts.notna().all().all(), "Every cleaned face entry must have a parseable source-page list."

manually_excluded_page_ids = {Path(filename).stem for filename in MANUALLY_EXCLUDED_FILENAMES}
referenced_page_ids = filename_parts.apply(
    lambda row: {f"{row['issue_id']}-{page_number}" for page_number in row['source_pages'].split(',')},
    axis=1,
)
manual_exclusion_mask = referenced_page_ids.map(lambda page_ids: bool(page_ids & manually_excluded_page_ids))
excluded_faces = faces.loc[manual_exclusion_mask].copy()

if not excluded_faces.empty:
    excluded_faces["matched_manually_excluded_page_ids"] = referenced_page_ids.loc[manual_exclusion_mask].map(
        lambda page_ids: ";".join(sorted(page_ids & manually_excluded_page_ids))
    )
    excluded_faces["action"] = "remove_face_entry_for_manually_excluded_image"
    assert not manual_exclusion_audit_csv.exists(), f"Manual-exclusion audit already exists: {manual_exclusion_audit_csv}"
    excluded_faces.to_csv(manual_exclusion_audit_csv, index=False)

    remaining_faces = faces.loc[~manual_exclusion_mask, original_columns].copy()
    remaining_faces.to_csv(cleaned_csv, index=False)
    reloaded_faces = pd.read_csv(cleaned_csv, dtype={"Filename": "string"})
    assert len(reloaded_faces) == len(faces) - len(excluded_faces)
    assert not reloaded_faces["Filename"].str.contains(
        re.escape("2006-0610-0118,0119_"), regex=True
    ).any()
else:
    assert manual_exclusion_audit_csv.exists(), (
        "No matching face entries remain and no prior manual-exclusion audit exists."
    )

print({
    "manually_excluded_dir": str(manually_excluded_dir),
    "moved_filenames": moved_filenames,
    "missing_image_filenames": missing_image_filenames,
    "removed_face_entries": len(excluded_faces),
    "manual_exclusion_audit_csv": str(manual_exclusion_audit_csv),
})

## Fixed inputs and outputs

Paths are relative to this notebook's `code/scripts` directory. The source directory is renamed only after all planned joins and image transfers have been verified.

In [ ]:
from pathlib import Path
import json
import re
import shutil

import pandas as pd
from PIL import Image

CLEANED_CSV = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_cleaned.csv")
DROPPED_CSV = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_dropped-over-two-pages.csv")
IMAGE_ROOT = Path("../../data/images")
SOURCE_IMAGE_DIR = IMAGE_ROOT / "full_pages_1940_2007"
JOINED_IMAGE_DIR = IMAGE_ROOT / "full_pages_1940_2007_joined"
UNUSED_IMAGE_DIR = IMAGE_ROOT / "full_pages_1940_2007_unused"
JOIN_MANIFEST_CSV = Path("../../data/processed/full_pages_1940_2007_joined_manifest.csv")
IMAGE_AUDIT_CSV = Path("../../data/processed/full_pages_1940_2007_joined_image_audit.csv")

for path in (CLEANED_CSV, DROPPED_CSV, SOURCE_IMAGE_DIR):
    assert path.exists(), f"Missing required input: {path}"

assert not JOINED_IMAGE_DIR.exists(), f"Joined output already exists: {JOINED_IMAGE_DIR}"
JOIN_MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "cleaned_csv": str(CLEANED_CSV),
    "dropped_csv": str(DROPPED_CSV),
    "source_image_dir": str(SOURCE_IMAGE_DIR),
    "joined_image_dir": str(JOINED_IMAGE_DIR),
    "unused_image_dir": str(UNUSED_IMAGE_DIR),
}, indent=2))

## Load and validate source-scan spans

A face filename begins with an issue identifier and an encoded comma-separated list of source pages. Multiple detected faces can belong to the same source scan, so the scan identifiers are deduplicated before planning image actions.

In [ ]:
FILENAME_PATTERN = r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)_"
PAGE_FILE_PATTERN = re.compile(r"^\d{4}-\d{4}-\d{4}\.jpg$")

cleaned_faces = pd.read_csv(CLEANED_CSV, usecols=["Filename"], dtype={"Filename": "string"})
dropped_faces = pd.read_csv(DROPPED_CSV, usecols=["Filename"], dtype={"Filename": "string"})

def unique_scans(frame, label):
    scan_parts = frame["Filename"].str.extract(FILENAME_PATTERN)
    assert scan_parts.notna().all().all(), f"Unparseable filenames in {label}"
    scans = scan_parts.drop_duplicates().copy()
    scans["page_numbers"] = scans["source_pages"].str.split(",")
    scans["page_count"] = scans["page_numbers"].str.len().astype("int64")
    scans["scan_id"] = scans["issue_id"] + "-" + scans["source_pages"]
    assert scans["scan_id"].is_unique
    return scans

cleaned_scans = unique_scans(cleaned_faces, "cleaned CSV")
dropped_scans = unique_scans(dropped_faces, "dropped CSV")

assert cleaned_scans["page_count"].le(2).all()
assert dropped_scans["page_count"].gt(2).all()
assert set(cleaned_scans["scan_id"]).isdisjoint(dropped_scans["scan_id"])

single_scans = cleaned_scans.loc[cleaned_scans["page_count"].eq(1)].copy()
two_page_scans = cleaned_scans.loc[cleaned_scans["page_count"].eq(2)].copy()

print(json.dumps({
    "single_page_scans": int(len(single_scans)),
    "two_page_scans": int(len(two_page_scans)),
    "over_two_page_scans": int(len(dropped_scans)),
    "largest_dropped_span": int(dropped_scans["page_count"].max()),
}, indent=2))

## Match every raw JPEG to an action

Every raw JPEG must be uniquely classified as a kept single page, a source page for a two-page join, or a page from a dropped scan. Two-page scans whose source JPEGs are partly or wholly absent are retained in the audit as missing-input records and do not produce an artificial joined image.

In [ ]:
def source_page_ids(scans):
    return {
        f"{scan.issue_id}-{page}"
        for scan in scans.itertuples(index=False)
        for page in scan.page_numbers
    }

source_raw_image_paths = sorted(path for path in SOURCE_IMAGE_DIR.glob("*.jpg") if PAGE_FILE_PATTERN.fullmatch(path.name))
unused_raw_image_paths = (
    sorted(path for path in UNUSED_IMAGE_DIR.glob("*.jpg") if PAGE_FILE_PATTERN.fullmatch(path.name))
    if UNUSED_IMAGE_DIR.exists() else []
)
source_raw_image_names = {path.name for path in source_raw_image_paths}
unused_raw_image_names = {path.name for path in unused_raw_image_paths}
assert source_raw_image_names.isdisjoint(unused_raw_image_names)
raw_image_names = source_raw_image_names | unused_raw_image_names
raw_page_ids = {Path(name).stem for name in raw_image_names}
assert len(raw_image_names) == len(raw_page_ids)

single_page_ids = source_page_ids(single_scans)
two_page_ids = source_page_ids(two_page_scans)
over_two_page_ids = source_page_ids(dropped_scans)

assert single_page_ids.isdisjoint(two_page_ids)
assert single_page_ids.isdisjoint(over_two_page_ids)
assert two_page_ids.isdisjoint(over_two_page_ids)
all_classified_page_ids = single_page_ids | two_page_ids | over_two_page_ids
assert raw_page_ids.issubset(all_classified_page_ids)
assert (all_classified_page_ids - raw_page_ids).issubset(two_page_ids)

two_page_scans["source_filenames"] = [
    [f"{scan.issue_id}-{page}.jpg" for page in scan.page_numbers]
    for scan in two_page_scans.itertuples(index=False)
]
two_page_scans["source_available"] = two_page_scans["source_filenames"].map(
    lambda names: all(name in raw_image_names for name in names)
)
two_page_scans["output_filename"] = two_page_scans["scan_id"].str.replace(",", "_", regex=False) + ".jpg"

available_two_page_scans = two_page_scans.loc[two_page_scans["source_available"]].copy()
missing_two_page_scans = two_page_scans.loc[~two_page_scans["source_available"]].copy()
assert available_two_page_scans["output_filename"].is_unique

print(json.dumps({
    "raw_jpegs": len(raw_image_names),
    "kept_single_jpegs": len(raw_page_ids & single_page_ids),
    "two_page_source_jpegs": len(raw_page_ids & two_page_ids),
    "over_two_page_jpegs": len(raw_page_ids & over_two_page_ids),
    "available_two_page_joins": len(available_two_page_scans),
    "missing_two_page_joins": len(missing_two_page_scans),
}, indent=2))
missing_two_page_scans[["scan_id", "source_filenames"]]

## Build reproducible manifests

The join manifest describes the images retained in the final folder. The image audit records the disposition of every available raw JPEG and any missing two-page source files.

In [ ]:
join_manifest = pd.concat([
    pd.DataFrame({
        "output_filename": sorted(f"{page_id}.jpg" for page_id in single_page_ids),
        "output_type": "single",
        "source_filenames": sorted(f"{page_id}.jpg" for page_id in single_page_ids),
        "scan_id": sorted(single_page_ids),
    }),
    available_two_page_scans[["output_filename", "scan_id", "source_filenames"]].assign(output_type="joined_two_page"),
], ignore_index=True)
join_manifest = join_manifest[["output_filename", "output_type", "source_filenames", "scan_id"]].sort_values("output_filename").reset_index(drop=True)
assert join_manifest["output_filename"].is_unique

audit_rows = []
available_two_page_ids = source_page_ids(available_two_page_scans)
missing_two_page_ids = source_page_ids(missing_two_page_scans)

for filename in sorted(raw_image_names):
    page_id = Path(filename).stem
    if page_id in single_page_ids:
        action, destination = "keep_single", JOINED_IMAGE_DIR / filename
    elif page_id in available_two_page_ids:
        action, destination = "copy_to_unused_then_remove_after_join", UNUSED_IMAGE_DIR / filename
    elif page_id in missing_two_page_ids:
        action, destination = "move_to_unused_missing_two_page_source", UNUSED_IMAGE_DIR / filename
    else:
        action, destination = "move_to_unused_over_two_pages", UNUSED_IMAGE_DIR / filename
    audit_rows.append({
        "raw_filename": filename,
        "scan_id": None,
        "action": action,
        "source_exists": True,
        "destination": str(destination),
    })

for scan in missing_two_page_scans.itertuples(index=False):
    for filename in scan.source_filenames:
        if filename in raw_image_names:
            continue
        audit_rows.append({
            "raw_filename": filename,
            "scan_id": scan.scan_id,
            "action": "missing_two_page_source_no_output",
            "source_exists": False,
            "destination": None,
        })

image_audit = pd.DataFrame(audit_rows).sort_values(["source_exists", "raw_filename"], ascending=[False, True]).reset_index(drop=True)
assert image_audit.loc[image_audit["source_exists"], "raw_filename"].is_unique
assert len(join_manifest) == len(single_page_ids) + len(available_two_page_scans)

display(image_audit["action"].value_counts().rename_axis("action").reset_index(name="image_rows"))
display(join_manifest["output_type"].value_counts().rename_axis("output_type").reset_index(name="output_images"))

## Join, transfer, and rename

Each join uses the page order encoded in the face filename. If source heights differ, the later page is resampled to the first page's height before the images are placed side by side. The original two-page JPEGs are copied to the unused folder and removed only after their joined output has been created and checked.

This cell changes the image directories. Run it only after reviewing the preflight counts above.

In [ ]:
def join_two_pages(source_paths, output_path):
    images = []
    for path in source_paths:
        with Image.open(path) as image:
            images.append(image.copy())

    target_height = images[0].height
    resized = []
    for image in images:
        if image.height != target_height:
            target_width = round(image.width * target_height / image.height)
            image = image.resize((target_width, target_height), Image.Resampling.LANCZOS)
        resized.append(image)

    output_mode = "L" if all(image.mode == "L" for image in resized) else "RGB"
    converted = [image.convert(output_mode) for image in resized]
    background = 255 if output_mode == "L" else (255, 255, 255)
    joined = Image.new(output_mode, (sum(image.width for image in converted), target_height), background)

    x_offset = 0
    for image in converted:
        joined.paste(image, (x_offset, 0))
        x_offset += image.width

    joined.save(output_path, format="JPEG", quality=95, subsampling=0, optimize=True, progressive=True, dpi=(300, 300))

UNUSED_IMAGE_DIR.mkdir(exist_ok=True)

def raw_source_path(filename):
    source_path = SOURCE_IMAGE_DIR / filename
    if source_path.exists():
        return source_path
    unused_path = UNUSED_IMAGE_DIR / filename
    assert unused_path.exists(), f"Missing raw source JPEG: {filename}"
    return unused_path

for scan in available_two_page_scans.itertuples(index=False):
    source_paths = [raw_source_path(filename) for filename in scan.source_filenames]
    output_path = SOURCE_IMAGE_DIR / scan.output_filename
    if output_path.exists():
        with Image.open(output_path) as existing_output:
            existing_output.verify()
    else:
        join_two_pages(source_paths, output_path)
    assert output_path.exists() and output_path.stat().st_size > 0

for scan in available_two_page_scans.itertuples(index=False):
    for filename in scan.source_filenames:
        source_path = SOURCE_IMAGE_DIR / filename
        unused_path = UNUSED_IMAGE_DIR / filename
        if source_path.exists():
            if unused_path.exists():
                assert unused_path.stat().st_size == source_path.stat().st_size
            else:
                shutil.copy2(source_path, unused_path)
            source_path.unlink()
        assert unused_path.exists()

for page_id in sorted(missing_two_page_ids & raw_page_ids):
    source_path = SOURCE_IMAGE_DIR / f"{page_id}.jpg"
    unused_path = UNUSED_IMAGE_DIR / source_path.name
    if source_path.exists():
        if unused_path.exists():
            assert unused_path.stat().st_size == source_path.stat().st_size
            source_path.unlink()
        else:
            shutil.move(source_path, unused_path)
    assert unused_path.exists()

for page_id in sorted(over_two_page_ids):
    source_path = SOURCE_IMAGE_DIR / f"{page_id}.jpg"
    unused_path = UNUSED_IMAGE_DIR / source_path.name
    if source_path.exists():
        if unused_path.exists():
            assert unused_path.stat().st_size == source_path.stat().st_size
            source_path.unlink()
        else:
            shutil.move(source_path, unused_path)
    assert unused_path.exists()

expected_joined_names = set(join_manifest["output_filename"])
expected_unused_names = {f"{page_id}.jpg" for page_id in two_page_ids | over_two_page_ids} & raw_image_names
actual_joined_names = {path.name for path in SOURCE_IMAGE_DIR.glob("*.jpg")}
actual_unused_names = {path.name for path in UNUSED_IMAGE_DIR.glob("*.jpg")}
assert actual_joined_names == expected_joined_names
assert actual_unused_names == expected_unused_names

SOURCE_IMAGE_DIR.rename(JOINED_IMAGE_DIR)
assert JOINED_IMAGE_DIR.exists() and not SOURCE_IMAGE_DIR.exists()

print(json.dumps({
    "joined_images": len(actual_joined_names),
    "unused_raw_images": len(actual_unused_names),
    "joined_folder": str(JOINED_IMAGE_DIR),
    "unused_folder": str(UNUSED_IMAGE_DIR),
}, indent=2))

## Write and verify manifests

The manifests allow the joined image set to be audited and rebuilt from the unused raw images and the two CSV inputs.

In [ ]:
join_manifest.to_csv(JOIN_MANIFEST_CSV, index=False)
image_audit.to_csv(IMAGE_AUDIT_CSV, index=False)

written_manifest = pd.read_csv(JOIN_MANIFEST_CSV)
written_audit = pd.read_csv(IMAGE_AUDIT_CSV)
final_jpegs = {path.name for path in JOINED_IMAGE_DIR.glob("*.jpg")}
unused_jpegs = {path.name for path in UNUSED_IMAGE_DIR.glob("*.jpg")}

assert len(written_manifest) == len(join_manifest)
assert len(written_audit) == len(image_audit)
assert set(written_manifest["output_filename"]) == final_jpegs
assert unused_jpegs == expected_unused_names
assert len(final_jpegs) + len(unused_jpegs) == len(raw_image_names) + len(available_two_page_scans)

summary = {
    "raw_input_jpegs": len(raw_image_names),
    "single_pages_kept_unchanged": len(single_page_ids),
    "two_page_images_joined": len(available_two_page_scans),
    "two_page_joins_missing_source_images": len(missing_two_page_scans),
    "over_two_page_raw_images_moved_to_unused": len(raw_page_ids & over_two_page_ids),
    "final_joined_images": len(final_jpegs),
    "unused_raw_images": len(unused_jpegs),
    "join_manifest_csv": str(JOIN_MANIFEST_CSV),
    "image_audit_csv": str(IMAGE_AUDIT_CSV),
}
print(json.dumps(summary, indent=2))
display(written_manifest["output_type"].value_counts().rename_axis("output_type").reset_index(name="output_images"))

In [ ]:
# One-off repair: 1996-0210-0123 was downloaded as a PNG, not a JPEG.
from pathlib import Path
import shutil

import pandas as pd
from PIL import Image

image_root = Path("../../data/images")
joined_dir = image_root / "full_pages_1940_2007_joined"
unused_dir = image_root / "full_pages_1940_2007_unused"
manifest_csv = Path("../../data/processed/full_pages_1940_2007_joined_manifest.csv")
audit_csv = Path("../../data/processed/full_pages_1940_2007_joined_image_audit.csv")

png_path = joined_dir / "1996-0210-0123.png"
jpg_path = unused_dir / "1996-0210-0124.jpg"
output_filename = "1996-0210-0123_0124.jpg"
output_path = joined_dir / output_filename
scan_id = "1996-0210-0123,0124"

assert png_path.exists(), png_path
assert jpg_path.exists(), jpg_path
assert not output_path.exists(), output_path

with Image.open(png_path) as png_image, Image.open(jpg_path) as jpg_image:
    left = png_image.convert("RGB")
    right = jpg_image.convert("RGB")

    if right.height != left.height:
        resized_width = round(right.width * left.height / right.height)
        right = right.resize((resized_width, left.height), Image.Resampling.LANCZOS)

    joined = Image.new("RGB", (left.width + right.width, left.height), "white")
    joined.paste(left, (0, 0))
    joined.paste(right, (left.width, 0))
    joined.save(output_path, format="JPEG", quality=95, subsampling=0, optimize=True, progressive=True, dpi=(300, 300))

assert output_path.exists() and output_path.stat().st_size > 0
shutil.move(png_path, unused_dir / png_path.name)

manifest = pd.read_csv(manifest_csv)
assert output_filename not in set(manifest["output_filename"])
manifest = pd.concat([
    manifest,
    pd.DataFrame([{
        "output_filename": output_filename,
        "output_type": "joined_two_page",
        "source_filenames": str([png_path.name, jpg_path.name]),
        "scan_id": scan_id,
    }]),
], ignore_index=True).sort_values("output_filename").reset_index(drop=True)
manifest.to_csv(manifest_csv, index=False)

audit = pd.read_csv(audit_csv)
audit = audit.loc[~audit["raw_filename"].eq("1996-0210-0123.jpg")].copy()
partner_mask = audit["raw_filename"].eq(jpg_path.name)
assert partner_mask.sum() == 1
audit.loc[partner_mask, ["scan_id", "action"]] = [scan_id, "copy_to_unused_then_remove_after_join"]
audit = pd.concat([
    audit,
    pd.DataFrame([{
        "raw_filename": png_path.name,
        "scan_id": scan_id,
        "action": "move_png_to_unused_after_join",
        "source_exists": True,
        "destination": str(unused_dir / png_path.name),
    }]),
], ignore_index=True).sort_values(["source_exists", "raw_filename"], ascending=[False, True]).reset_index(drop=True)
audit.to_csv(audit_csv, index=False)

assert len(list(joined_dir.glob("*.jpg"))) == len(manifest)
assert (unused_dir / png_path.name).exists()
assert not png_path.exists()
print(f"Wrote {output_path} and updated the manifest and audit.")